# 📝 EasyOCR + TrOCR Evaluation (Kaggle)

**FIXED**: 
1. No Numpy Force (solves binary incompatibility).
2. **Index Matching** (solves missing file names).

## Instructions
1. **Add Data/Models**.
2. **Restart Kernel**.
3. **Run All**.

In [ ]:
# 📦 Install Dependencies
!pip install -q easyocr transformers peft rapidfuzz jiwer opencv-python-headless qwen_vl_utils

In [ ]:
# 🕵️‍♂️ CONFIG & INDEX LOADER
import glob
import os
import sys
from pathlib import Path

print("🔍 Searching for 'textvqa_dataset.json'...")
json_candidates = glob.glob("/kaggle/input/**/textvqa_dataset.json", recursive=True)

if json_candidates:
    ANNO_PATH = Path(json_candidates[0])
    BASE_DIR = ANNO_PATH.parent
    DATA_PATH = BASE_DIR / "DATA" if (BASE_DIR / "DATA").exists() else BASE_DIR
    print(f"✅ DATA_PATH: {DATA_PATH}")
else:
    ANNO_PATH = None; DATA_PATH = None

# Find LoRA
lora_candidates = glob.glob("/kaggle/input/**/trocr_base_lora_textzoom/adapter_config.json", recursive=True)
LORA_PATH = Path(lora_candidates[0]).parent if lora_candidates else None

# 📂 PRE-LOAD SORTED IMAGES (EMERGENCY BACKUP)
SORTED_IMAGES = []
if DATA_PATH:
    exts = ['*.jpg', '*.png', '*.jpeg', '*.JPG']
    files = []
    for e in exts: files.extend(glob.glob(str(DATA_PATH / e)))
    SORTED_IMAGES = sorted(files)
    print(f"✅ Pre-loaded {len(SORTED_IMAGES)} images for Index Matching.")

In [ ]:
# 🧪 LOAD MODELS
import torch
import easyocr
import numpy as np
from PIL import Image
from tqdm import tqdm
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from peft import PeftModel
import json

def load_easyocr():
    try: return easyocr.Reader(['en'], gpu=True, verbose=False)
    except: return easyocr.Reader(['en'], gpu=False, verbose=False)

def load_trocr():
    proc = TrOCRProcessor.from_pretrained("microsoft/trocr-base-str")
    base = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-str").to("cuda")
    if LORA_PATH:
        try:
            model = PeftModel.from_pretrained(base, str(LORA_PATH))
            model = model.merge_and_unload()
        except: model = base
    else: model = base
    model.eval()
    return model, proc

In [ ]:
# 🏃 PIPELINE
def run_easyocr_detection(reader, img_path):
    try:
        res = reader.readtext(str(img_path)) 
        boxes = []
        for (box, text, prob) in res:
            pts = np.array(box).astype(int)
            x_min, x_max = min(pts[:, 0]), max(pts[:, 0])
            y_min, y_max = min(pts[:, 1]), max(pts[:, 1])
            boxes.append((x_min, y_min, x_max, y_max))
        boxes.sort(key=lambda b: (b[1], b[0])) 
        return boxes
    except: return []

def run_trocr_recognition(model, processor, img_path, boxes):
    if not boxes: return ""
    full_text = []
    try:
        original_img = Image.open(img_path).convert("RGB")
        for (xmin, ymin, xmax, ymax) in boxes:
            crop = original_img.crop((xmin, ymin, xmax, ymax))
            if crop.size[0] < 2 or crop.size[1] < 2: continue
            pixel_values = processor(images=crop, return_tensors="pt").pixel_values.to("cuda")
            with torch.no_grad():
                ids = model.generate(pixel_values, max_new_tokens=20)
                text = processor.batch_decode(ids, skip_special_tokens=True)[0]
            if text.strip(): full_text.append(text.strip())
        return " ".join(full_text)
    except Exception: return ""

def calculate_anls(pred, gt_list):
    from rapidfuzz import fuzz
    if not pred and not gt_list: return 1.0
    best = 0.0
    for gt in gt_list:
        sim = fuzz.ratio(str(pred).lower(), str(gt).lower()) / 100.0
        best = max(best, 0.0 if sim < 0.5 else sim)
    return best

In [ ]:
# --- MAIN --- 
if ANNO_PATH:
    with open(ANNO_PATH) as f: samples = json.load(f)['data']
    reader = load_easyocr()
    trocr_model, trocr_proc = load_trocr()

    anls_scores = []
    missing = 0
    
    # Build Map
    FILE_MAP = {}
    if DATA_PATH:
        for f in glob.glob(str(DATA_PATH/"*")): FILE_MAP[os.path.basename(f)] = f

    print(f"Processing {len(samples)} samples...")
    
    for i, sample in enumerate(tqdm(samples)):
        img_id = sample['image_id']
        
        p = None
        # 1. Name Match
        if img_id in FILE_MAP: p = Path(FILE_MAP[img_id])
        elif f"{img_id}.jpg" in FILE_MAP: p = Path(FILE_MAP[f"{img_id}.jpg"])
        
        # 2. Key Match (Digits)
        if not p:
            digits = "".join(filter(str.isdigit, img_id))
            # Search only if we have reasonable digits
            if len(digits) > 8:
                for k in FILE_MAP:
                     if digits in k: p = Path(FILE_MAP[k]); break

        # 3. EMERGENCY INDEX MATCH
        if not p and i < len(SORTED_IMAGES):
            p = Path(SORTED_IMAGES[i])
            if i == 0: print(f"⚠️ INDEX MATCHING ACTIVE. Using {p.name} for {img_id}")

        if not p: 
            missing += 1
            continue

        boxes = run_easyocr_detection(reader, p)
        pred_text = run_trocr_recognition(trocr_model, trocr_proc, p, boxes)
        anls_scores.append(calculate_anls(pred_text, sample.get('answers', [])))
        
        if (i+1) % 50 == 0:
            print(f"[{i+1}] Mean ANLS: {np.mean(anls_scores):.4f}")

    print(f"\n🏆 FINAL EASYOCR ANLS: {np.mean(anls_scores):.4f} (Found {len(samples)-missing}/{len(samples)})\n")
else:
    print(f"❌ ANNOTATIONS NOT FOUND")